# BioNodulo on Google Colab

This notebook launches a temporary BioNodulo instance inside a Colab runtime. Files created in Colab are ephemeral unless you download or save them elsewhere.

BioNodulo is distributed under the BioNodulo Research License. Publication, commercial use, and hosted services require a separate license.

In [ ]:
%cd /content
!test -d BioNodulo || git clone -q --branch bionodulo-collab https://github.com/Classacre/BioNodulo.git
%cd /content/BioNodulo
!git fetch -q origin bionodulo-collab
!git checkout -q bionodulo-collab
!git pull -q --ff-only origin bionodulo-collab
!python -m pip install -q .

## Start BioNodulo

Run this cell to embed BioNodulo below and print a Colab-proxied fallback link.

Use the embedded app output below. If your browser blocks the embed, open the Colab-proxied BioNodulo link printed above it. Do not use a `https://localhost:8000/` link from an old Colab output cell.

In [ ]:
import subprocess
import sys
import time
from html import escape
from urllib.request import urlopen
from pathlib import Path
from google.colab import output
from IPython.display import HTML, display

workspace = Path('/content/bionodulo_workspace')
workspace.mkdir(exist_ok=True)

def bionodulo_ready():
    try:
        with urlopen('http://127.0.0.1:8000/api/health', timeout=1) as response:
            return response.status == 200
    except Exception:
        return False

if not bionodulo_ready():
    server = subprocess.Popen([
        sys.executable,
        'main.py',
        '--host', '0.0.0.0',
        '--port', '8000',
        '--project-root', str(workspace),
    ])

for _ in range(60):
    if bionodulo_ready():
        break
    else:
        time.sleep(1)
else:
    raise RuntimeError('BioNodulo did not become ready on port 8000.')

proxy_url = output.eval_js('google.colab.kernel.proxyPort(8000)')
safe_url = escape(proxy_url)
display(HTML(f'''
<p><a href="{safe_url}" target="_blank" rel="noreferrer">Open BioNodulo in a Colab-proxied tab</a></p>
<iframe src="{safe_url}" width="100%" height="900" style="border:0;border-radius:8px" allow="clipboard-read; clipboard-write"></iframe>
'''))